In [ ]:
#include <Wire.h>

/* -------- MOTOR PINS -------- */
#define M1A 6
#define M1B 9
#define M2A 10
#define M2B 11

/* -------- ENCODERS -------- */
#define ENCODER1_A 2
#define ENCODER1_B 4
#define ENCODER2_A 3
#define ENCODER2_B 5

volatile long count1 = 0;
volatile long count2 = 0;

void encoder1ISR() {
    if (digitalRead(ENCODER1_B))
        count1--;
    else
        count1++;
}

void encoder2ISR() {
    if (digitalRead(ENCODER2_B))
        count2++;
    else
        count2--;
}

/* -------- MPU6500 -------- */
#define MPU_ADDR 0x68

int16_t ax, ay, az, gx, gy, gz;

float pitch = 0;
unsigned long lastTime = 0;

#define ALPHA 0.98

void initMPU() {
    Wire.beginTransmission(MPU_ADDR);
    Wire.write(0x6B);
    Wire.write(0x00);
    Wire.endTransmission();
}

void readMPU() {
    Wire.beginTransmission(MPU_ADDR);
    Wire.write(0x3B);
    Wire.endTransmission(false);
    Wire.requestFrom(MPU_ADDR, 14, true);

    ax = Wire.read() << 8 | Wire.read();
    ay = Wire.read() << 8 | Wire.read();
    az = Wire.read() << 8 | Wire.read();

    Wire.read();
    Wire.read();

    gx = Wire.read() << 8 | Wire.read();
    gy = Wire.read() << 8 | Wire.read();
    gz = Wire.read() << 8 | Wire.read();
}

float getPitchAccel() {
    return atan2((float)ay, (float)az) * 180.0 / PI;
}

float getGyroRate() {
    return (float)gx / 131.0;
}

void updatePitch() {
    unsigned long now = millis();
    float dt = (now - lastTime) / 1000.0;
    lastTime = now;

    readMPU();

    float accel = getPitchAccel();
    float gyro = getGyroRate();

    pitch = ALPHA * (pitch + gyro * dt)
          + (1 - ALPHA) * accel;

    // Smooth value for cleaner graph
    pitch = round(pitch * 10) / 10.0;
}

/* -------- PID -------- */
float Kp = 5;
float Ki = 7;
float Kd = 2;  // Report also mentions 2.1

float setpoint = -4;
float error, prevError = 0;
float integral = 0.80;
float output;

/* -------- MOTOR SPEED TRACK -------- */
int leftSpeed = 0;
int rightSpeed = 0;

/* -------- MOTOR CONTROL -------- */
void setMotor(int speed) {
    speed = constrain(speed, -255, 255);

    leftSpeed = speed;
    rightSpeed = speed;

    if (speed > 0) {
        analogWrite(M1A, speed);
        analogWrite(M1B, 0);

        analogWrite(M2A, 0);
        analogWrite(M2B, speed);
    }
    else {
        int s = -speed;

        analogWrite(M1A, 0);
        analogWrite(M1B, s);

        analogWrite(M2A, s);
        analogWrite(M2B, 0);
    }
}

/* -------- SETUP -------- */
void setup() {
    Serial.begin(9600);

    pinMode(M1A, OUTPUT);
    pinMode(M1B, OUTPUT);
    pinMode(M2A, OUTPUT);
    pinMode(M2B, OUTPUT);

    pinMode(ENCODER1_A, INPUT_PULLUP);
    pinMode(ENCODER1_B, INPUT_PULLUP);
    pinMode(ENCODER2_A, INPUT_PULLUP);
    pinMode(ENCODER2_B, INPUT_PULLUP);

    attachInterrupt(
        digitalPinToInterrupt(ENCODER1_A),
        encoder1ISR,
        RISING
    );

    attachInterrupt(
        digitalPinToInterrupt(ENCODER2_A),
        encoder2ISR,
        RISING
    );

    Wire.begin();
    initMPU();
    delay(100);

    readMPU();
    pitch = getPitchAccel();
    lastTime = millis();

    Serial.println("Pitch,Setpoint,Upper,Lower,Output");
}

/* -------- LOOP -------- */
void loop() {
    updatePitch();

    // -------- DEAD ZONE --------
    if (pitch > -4 && pitch < -3) {
        error = 0;
        integral = 0;
        output = 0;

        setMotor(0);
    }
    else {
        error = pitch - setpoint;
        integral += error;

        // Anti-windup
        integral = constrain(integral, -100, 100);

        float derivative = error - prevError;

        output = Kp * error
               + Ki * integral
               + Kd * derivative;

        prevError = error;

        setMotor(-output);
    }

    // -------- SERIAL GRAPH OUTPUT --------
    Serial.print(pitch);
    Serial.print(",");
    Serial.print(setpoint);
    Serial.print(",");
    Serial.print(2);
    Serial.print(",");
    Serial.print(-2);
    Serial.print(",");
    Serial.println(output);

    delay(5);
}